In [ ]:
import sys
sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, mesh
from numpy import linalg as la

In [ ]:
F = np.array([[ 0.54606617, -0.22851915],
       [-0.59135983,  0.22277653],
       [-0.67489594,  0.63440734]])

In [ ]:
U, S, Vt = la.svd(F)
dF = U @ [[0, -1], [1, 0], [0, 0]] @ Vt

eps = 1e-7
U_p, S_p, Vt_p = la.svd(F + eps * dF)
U_m, S_m, Vt_m = la.svd(F - eps * dF)
U_dot = (U_p - U_m) / (2 * eps)
S_dot = (S_p - S_m) / (2 * eps)
Vt_dot = (Vt_p - Vt_m) / (2 * eps)
print(U_dot)
print(S_dot)
print(Vt_dot)

In [ ]:
U_dot @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]]) @ U.transpose() + U @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]]) @ U_dot.transpose()

In [ ]:
U.transpose() @ U_dot

In [ ]:
Vt.transpose() @ Vt_dot

In [ ]:
def optionalRandom32(A = None):
    if (A is not None): return A
    return np.random.uniform(low=-1, high=1, size=(3, 2))
def compare(F = None, dF = None, dF2 = None, eps=1e-8):
    F = optionalRandom32(F)
    dF = optionalRandom32(dF)
    dF2 = optionalRandom32(dF2)
    
#     U, S, Vt = la.svd(F)
#     dF = U @ [[0, 0], [1, 0], [0, 0]] @ Vt
#     dF2 = dF
    
    dC = dF.transpose() @ F + F.transpose() @ dF
    dC2 = dF2.transpose() @ F + F.transpose() @ dF2
    d2C = dF.transpose() @ dF2 + dF2.transpose() @ dF
    
    tfe = inflation.OptionalTensionFieldEnergy(F.transpose() @ F)
    tfe.useTensionField = False
    psi = inflation.IncompressibleBalloonEnergyWithHessProjection(F)
    def energyAt(FF):
        psi.setF(FF)
        e = psi.energy()
        psi.setF(F)
        return e
    print(f'{tfe.energy()}\t{psi.energy()}')
    print(f'{tfe.denergy(dC)}\t{psi.denergy(dF)}\t{(psi.denergy().transpose() @ dF).trace()}')
    print(f'{tfe.d2energy(dC, dC2) + tfe.denergy(d2C)}\t{psi.d2energy(dF, dF2)}')
    return
    for probe_i in range(3):
        for probe_j in range(2):
            Fdot = np.outer(U[:, probe_i], Vt[probe_j, :])
            print(f'({probe_i},{probe_j}) energy derivative: {psi.denergy(Fdot)}')
    stretch_modes = [(0, 0), (1, 1)]
    for i, di in enumerate(stretch_modes):
        dFi = np.outer(U[:, di[0]], Vt[di[1], :])
        for j, dj in enumerate(stretch_modes):
            dFj = np.outer(U[:, dj[0]], Vt[dj[1], :])
            print(f'(d{i}, d{j}): {psi.d2energy(dFi, dFj)}')
compare()

In [ ]:
compare(F)

In [ ]:
compare(F)

In [ ]:
import sys
sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, mesh

In [ ]:
m = mesh.Mesh('../../examples/single_tri.obj')
isheet = inflation.InflatableSheet(m)

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full
isheet.pressure = 1

In [ ]:
# perturbation putting the single triangle example in a partial tension state
eval_perturb  = np.array([-0.25989305, -0.36850698, -0.9096837 , -0.42756298, -0.2256421,  -0.83363732,  0.51394147, -0.60510383, -0.43363783])

In [ ]:
fd_perturb = np.random.uniform(low=-1.0, high=1.0, size=isheet.numVars())

In [ ]:
xorig = isheet.getVars()
xeval = isheet.getVars() + 1e-3 * eval_perturb

In [ ]:
# Set current point as "x_eval" and verify that tension state doesn't change in the surrounding neighborhood 
isheet.setVars(xeval + 1e-6 * fd_perturb)
print(isheet.tensionStateHistogram())
isheet.setVars(xeval - 1e-6 * fd_perturb)
print(isheet.tensionStateHistogram())
isheet.setVars(xeval)
print(isheet.tensionStateHistogram())

In [ ]:
isheet.setUseTensionFieldEnergy(False)
isheet.setUseHessianProjectedEnergy(True)

In [ ]:
fd_validation.validateHessian(isheet, xeval=isheet.getVars(), fd_eps=1e-7,
                              etype=Elastic)